In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

In [2]:
df = pd.read_csv('../../datasets/diabetes.csv')
df.sample(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
572,3,111,58,31,44,29.5,0.430,22,0
670,6,165,68,26,168,33.6,0.631,49,0
53,8,176,90,34,300,33.7,0.467,58,1
635,13,104,72,0,0,31.2,0.465,38,1
312,2,155,74,17,96,26.6,0.433,27,1
128,1,117,88,24,145,34.5,0.403,40,1
545,8,186,90,35,225,34.5,0.423,37,1
425,4,184,78,39,277,37.0,0.264,31,1
192,7,159,66,0,0,30.4,0.383,36,1
30,5,109,75,26,0,36.0,0.546,60,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
X = df.iloc[: , :-1].values
y = df.iloc[: , -1].values

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
import tensorflow as tf 
from tensorflow import keras
from keras.layers import Dense,Input
from keras.models import Sequential
from keras_tuner import RandomSearch

In [13]:
X_train.shape 

(614, 8)

### Tunning Hyperparamater Optimizer

In [14]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape = (X_train.shape[1],)))
    model.add(Dense(32 , activation='relu'))
    model.add(Dense(1 , activation = 'sigmoid'))
    
    optimizer = hp.Choice('optimizer' , ['adam','sdg','rmsprop','adagrad'])

    model.compile(optimizer=optimizer , loss = 'binary_crossentropy' ,metrics=['accuracy'])

    return model

In [18]:
tuner = RandomSearch(
    build_model , 
    objective = 'val_accuracy' , 
    max_trials = 4 , 
    executions_per_trial = 2 , 
    directory = 'tuner_dir' , 
    project_name = 'optimizer_tunning'
)
tuner.search(X_train, y_train , epochs = 7 , validation_data = (X_test , y_test))

Trial 4 Complete [00h 00m 06s]
val_accuracy: 0.7532467544078827

Best val_accuracy So Far: 0.7532467544078827
Total elapsed time: 00h 00m 19s


In [23]:
best_optimizer = tuner.get_best_hyperparameters()[0].get('optimizer')
print(f'Best optimizer: {best_optimizer}')

Best optimizer: adam


In [ ]:
best_model = tuner.get_best_models(num_models = 1)[0]
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f"Test accuracy: {accuracy:.4f}")

c:\Users\Sachin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7250 - loss: 0.5333  
Test accuracy: 0.7662


In [24]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

### Tunning No of neurons in each layer

In [31]:
def build_model2(hp):
    model = Sequential()

    model.add(Input(shape = (X_train.shape[1],))) , 
    model.add(Dense(
        units = hp.Int('units' , min_value = 8 , max_value = 128 , step = 8) , 
        activation = 'relu' 
    ))
    model.add(Dense(1 , activation = 'sigmoid'))

    model.compile(
        optimizer = 'adam' , 
        loss = 'binary_crossentropy' ,
        metrics=['accuracy']
    )

    return model 

In [33]:
tuner2 = RandomSearch(
    build_model2 , 
    objective = 'val_accuracy' ,
    max_trials = 4 ,
    directory = 'tuner_dir' ,
    project_name = 'units_tunning'
)
tuner2.search(X_train , y_train , epochs = 7 , validation_data = (X_test , y_test))

Trial 4 Complete [00h 00m 03s]
val_accuracy: 0.7662337422370911

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 13s


In [38]:
tuner2.get_best_hyperparameters()[0].values['units']

96

In [39]:
best_model2 = tuner2.get_best_models(num_models = 1)[0]
best_model2.summary()

c:\Users\Sachin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 96)             │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            97 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 961 (3.75 KB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 0 (0.00 B)

### Tunning No of layers

In [40]:
def build_model3(hp):
    model = Sequential()
    model.add(Input(shape = (X_train.shape[1],))) , 
    
    for i in range(hp.Int('num_layers' , min_value = 1 , max_value = 10)):
        model.add(Dense(96, activation='relu'))

    model.add(Dense(1 , activation = 'sigmoid'))
    model.compile(
        optimizer = 'adam' , 
        loss = 'binary_crossentropy' ,
        metrics=['accuracy']
    )
    return model 

In [50]:
tuner3 = RandomSearch(
    build_model3 , 
    objective = 'val_accuracy' ,
    max_trials = 10 ,   # all 10 layers
    directory = 'tuner_dir' ,
    project_name = 'no_of_layers_tunning'
)
tuner3.search(X_train , y_train , epochs = 7 , validation_data = (X_test , y_test))

Trial 10 Complete [00h 00m 05s]
val_accuracy: 0.7792207598686218

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 02m 46s


In [51]:
tuner3.get_best_hyperparameters()[0].values

{'num_layers': 8}